# LLM smoke test

Checks the app's LLM wiring end to end.
Run the cells top to bottom; each one fails loudly with the specific thing to fix.

The app ships with two providers and a sidebar switch between them, defaulting to
DeepSeek V4 Flash. This notebook pins one explicitly in section 0 — set `PROVIDER`
there to whichever you want to exercise. Section 1 (model discovery) is Gemini's
API and is skipped for DeepSeek; everything else runs against either.

What it covers:

1. the provider's credentials are visible to the app
2. the model id is valid and answers
3. structured output works — this is what `sentiment.py` depends on
4. tool calling works — this is what the assistant agent depends on
5. the app's own entry points (`get_recent_sentiment`, `get_financial_agent`) run

**The model id is the likely failure.** Nothing in the app hard-codes it: if step 2
reports the model is unknown, change `GEMINI_MODEL_NAME` (or `DEEPSEEK_MODEL_NAME`)
in `.env` and re-run. Gemini's current model list is at
https://ai.google.dev/gemini-api/docs/models; the Azure deployment name is on the
deployment's page in AI Foundry.


## 0. Environment

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import config, llm as llm_factory
importlib.reload(config); importlib.reload(llm_factory)

# "deepseek" or "gemini". Pinned rather than inherited from LLM_PROVIDER so that
# a run of this notebook tests the provider you meant, not whichever one .env
# happens to name today.
PROVIDER = "deepseek"
llm_factory.set_provider(PROVIDER)

print(f"repo root : {REPO_ROOT}")
print(f"provider  : {llm_factory.describe()}")

if not llm_factory.credentials_present():
    raise SystemExit(
        llm_factory.missing_credentials_message()
        + " Then restart the kernel — python-dotenv reads .env at import time."
    )

## 1. Which models does this key actually have? *(Gemini only)*

Run this **first** when anything fails. Model names move faster than any list
written into a repo, so this asks Google rather than assuming, and tells you
straight away whether `GEMINI_MODEL_NAME` is a real id for your key.

Same thing from a shell, without the notebook: `python llm.py`

In [ ]:
if PROVIDER == "gemini":
    print(llm_factory.render_models(llm_factory.list_models()))
else:
    # Azure AI Foundry has no equivalent free listing call; the deployment name
    # is whatever the portal shows on the deployment's own page.
    print("Skipped — model discovery is the Gemini API. Deployment in use:",
          llm_factory.active_model())

## 2. The model answers

The first real network call. A 404 or "model not found" here means the model id is
wrong for this provider, not that the credential is bad —
`GEMINI_MODEL_NAME` for Gemini, `DEEPSEEK_MODEL_NAME` for the Azure deployment.

In [ ]:
llm = llm_factory.get_chat_model(temperature=0.0)
try:
    reply = llm.invoke("Reply with exactly: pong")
    print("model replied:", reply.content.strip())
except Exception as exc:
    print(f"FAILED: {type(exc).__name__}: {exc}\n")
    print("If this says the model is unknown or not found, the id is wrong.")
    print(f"  currently: {config.GEMINI_MODEL_NAME}")
    print("  fix      : set GEMINI_MODEL_NAME in .env, restart the kernel, re-run")
    print("  list     : https://ai.google.dev/gemini-api/docs/models")
    raise

## 3. Structured output

`sentiment.py` calls `with_structured_output(SentimentResult)`. Providers differ in
how well they honour a schema, so this is worth checking directly rather than
discovering it through an empty sentiment panel.

In [ ]:
from sentiment import SentimentResult

structured = llm_factory.get_chat_model(temperature=0.0).with_structured_output(SentimentResult)
result = structured.invoke([
    ("system", "You are a financial-news classification AI. Return structured JSON."),
    ("user", '{"ticker": "TEST", "news": [{"title": "TEST beats earnings, raises guidance"}]}'),
])

print(f"label      : {result.label}")
print(f"score      : {result.score:+.2f}")
print(f"confidence : {result.confidence:.0%}")
assert -1 <= result.score <= 1, "score outside the schema's declared range"
print("\nstructured output OK")

## 4. Tool calling

The assistant agent is a tool-calling loop. If the model will not emit a tool call,
the agent silently degrades into an ordinary chatbot that makes numbers up — which is
exactly what the tools exist to prevent.

In [ ]:
from langchain_core.tools import tool

@tool
def get_price(ticker: str) -> str:
    """Return the current price for a ticker."""
    return f"{ticker} is trading at 123.45"

bound = llm_factory.get_chat_model(temperature=0.0).bind_tools([get_price])
response = bound.invoke("What is NVDA trading at? Use the tool.")

if not response.tool_calls:
    print("NO TOOL CALL — the agent will not work with this model.")
    print("Response was:", response.content[:200])
    raise SystemExit("tool calling unsupported or refused")

print("tool calls:", [(c["name"], c["args"]) for c in response.tool_calls])
print("\ntool calling OK")

## 5. The app's own entry points

Same code paths the dashboard uses, so a pass here means the tabs will work.

In [ ]:
from sentiment import get_recent_sentiment

payload = get_recent_sentiment("NVDA", days=7)
if payload.get("error"):
    print(f"sentiment: {payload['error']}")
    print("(an empty news window is a normal result, not a failure)")
else:
    data = payload["data"]
    print(f"sentiment  : {data['label']} ({data['score']:+.2f}) "
          f"from {len(payload['articles'])} headlines")

In [ ]:
from chat_agent import get_financial_agent

agent = get_financial_agent()
if agent is None:
    raise SystemExit("agent not built — credentials_present() returned False")

answer = agent.invoke({
    "input": "Use check_signal_now to tell me the current breakout signal for AAPL.",
    "chat_history": [],
})["output"]
print(answer[:900])

## 6. Which tools the agent has

A sanity check on the wiring rather than the model. Thirteen tools is enough that
tool *selection* becomes its own failure mode — if the agent above reached for the
wrong one, this is the list to prune.

In [ ]:
import ast

tree = ast.parse((REPO_ROOT / "chat_agent.py").read_text())
names = [n.name for n in ast.walk(tree)
         if isinstance(n, ast.FunctionDef)
         and any(getattr(d, "id", "") == "tool" for d in n.decorator_list)]
print(f"{len(names)} tools registered:")
for n in sorted(names):
    print("  -", n)

---

## If something failed

**No credentials** — `.env` must sit at the repo root and be read before `config` is
imported. Restart the kernel after editing it; `python-dotenv` reads the file once at
import. `python llm.py` prints what the app actually loaded for the active provider.

**Model not found** — the id string is wrong for this provider. On Gemini, run step 1
(or `LLM_PROVIDER=gemini python llm.py`) for the list this key can actually call and
set `GEMINI_MODEL_NAME` to one of them. On DeepSeek, `DEEPSEEK_MODEL_NAME` must be the
*deployment* name from AI Foundry, which is not always the model's catalogue name.
Nothing else in the app hard-codes a model name.

**Structured output empty or malformed** — `sentiment.py` will report an LLM error
rather than a wrong score, so this degrades safely, but the sentiment panel stays blank.

**No tool call in step 3** — the assistant tab will still answer, but from the context
blob alone rather than by computing. That is the failure mode the tools were added to
remove, so treat it as blocking rather than cosmetic.

**Rate limits** — the Gemini free tier limits requests per minute, and Azure
deployments have their own per-minute token quota. A 429 here means wait, not
misconfiguration. Switching provider in the sidebar is the quick way around one.
